![DB Academy](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/common/db-academy.png)

# 13 - Continuous Integration and Continuous Deployment (CI/CD) with Declarative Automation Bundles (DABs)

## Overview

In this demonstration, you'll build on everything you've learned about DABs and apply it to a CI/CD workflow with three environments (`development`, `stage`, `production`). 

The bundle deploys a workflow that runs **unit tests**, a **Apache Spark™ Declarative Pipeline (SDP)** for ETL and integration tests, and a **visualization notebook**. Each target overrides the catalog, raw data path, and (for dev/stage) the compute, while production runs on Serverless.


## Learning Objectives

By the end of this demonstration, you will be able to:

1. **Read and reason about a multi-file bundle** that splits resources into per-asset YAML files and uses a dedicated `variables.yml`.
2. **Set bundle variables** (including a `lookup` variable that resolves a cluster name to a cluster ID).
3. **Deploy and run unit tests, a Apache Spark™ Declarative Pipeline, and a visualization** as a single workflow.
4. **Promote the same bundle across `development`, `stage`, and `production` targets** with per-target overrides for catalog, raw data path, and compute.
5. **Override variables from the CLI** with `databricks bundle <command> --var="<name>=<value>" -t <target>`.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT
<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select All-Purpose Compute</strong>
  <div style="color:#333;">

This notebook requires **all-purpose compute** (Dedicated). Serverless is not supported for this notebook.

Follow these steps to attach an all-purpose compute cluster:

1. Navigate to the top-right of this notebook and click the drop-down menu to select your `labuser_USERNAME` cluster.
    - By default, the notebook might use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

    - In the drop-down, select **More**.

    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.

⚠️ **NOTE:** If the cluster shows a **terminated** state (red dot in the cluster picker), it needs to be started before you can attach. Click the cluster, then **Start**, and wait a few minutes until you see a green dot.
  </div>
</div>


## REQUIRED - DATA SETUP

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Data Setup</strong>
  <div style="color:#333;">

Recall that your environment was setup using the **02 - REQUIRED - Course Setup and Authentication**.

If you end your lab or your lab session times out, your environment will be reset. You will need to rerun the **02 - REQUIRED - Course Setup and Authentication** notebook to recreate the catalogs and data for your environment.

  </div>
</div>




## A. Classroom Setup

Run the following cell to configure your working environment for this course. 

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

In [0]:
%run ../Includes/Classroom-Setup-13

Run the cell below to confirm the Databricks CLI is working.

In [0]:
%sh
databricks catalogs list

## B. Full Project Folder Architecture

### B1. Full Project Folder Architecture

Before deploying, it helps to see how the full project is organized. The bundle is built up step by step — click each step below to build the project folder structure and highlight what that step adds.

<div style="font-family:DM Sans,system-ui,sans-serif;max-width:1200px;margin:8px auto"><div style="text-align:center;max-width:1000px;margin:18px auto 6px"><div style="font-size:32px;font-weight:700;color:#0B2026">Full Project Outline</div><div style="font-size:22px;color:#1B5162;font-weight:500;margin-top:2px">Folder Architecture</div></div><div style="text-align:center;font-size:14px;color:#1B5162;font-weight:500;margin:2px 0 12px">Click a step to build the project folder structure — each step highlights what it adds.</div><div style="border:1px solid #E2E0DB;border-radius:12px;background:#fff;padding:10px 14px;max-width:1180px;margin:0 auto"><div id="fpo2-fr-s1" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s1" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s1" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s1" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" fill-opacity="1" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="1673.6" y="612.2" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1718.0" y="658.9" text-anchor="middle" font-size="40.0" fill="#F9F7F4" font-weight="700">1</text><path d="M1496.7,487.5 L1496.7,650.7 L1673.6,650.7" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"/><text x="1781.4" y="656.6" text-anchor="start" font-size="45.8" fill="#1B3139" font-weight="500">Create a project folder.</text></svg></div>
<div id="fpo2-fr-s2" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s2" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s2" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s2" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" fill-opacity="1" stroke="#1B5162" stroke-width="1.5"/><text x="1795.5" y="535.6" text-anchor="middle" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" fill-opacity="1" stroke="#1B5162" stroke-width="1.5"/><text x="385.6" y="1028.1" text-anchor="middle" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" fill-opacity="1" stroke="#1B5162" stroke-width="1.5"/><text x="385.6" y="609.2" text-anchor="middle" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="1214.9" y="923.8" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1259.3" y="970.5" text-anchor="middle" font-size="40.0" fill="#F9F7F4" font-weight="700">2</text><path d="M424.1,1261.0 L819.5,1261.0 L819.5,962.3 L1214.9,962.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-start="url(#cxnO-fpo2-s2)"/><path d="M434.6,800.6 L822.5,800.6 L822.5,962.3 L1210.3,962.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-start="url(#cxnO-fpo2-s2)"/><rect x="1259.3" y="1020.7" width="685.1" height="181.8" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1601.8" y="1065.8" text-anchor="middle" font-size="36.7" fill="#1B3139" font-weight="500">Further build project structure by</text><text x="1601.8" y="1108.0" text-anchor="middle" font-size="36.7" fill="#1B3139" font-weight="500">adding high-level directories to</text><text x="1601.8" y="1150.2" text-anchor="middle" font-size="36.7" fill="#1B3139" font-weight="500">isolate project components</text><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><path d="M1308.3,962.3 L1746.5,962.2" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s2)"/><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s2b" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s2b" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s2b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s2b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="336.6" y="1218.5" width="98.0" height="84.9" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="385.6" y="1265.2" text-anchor="middle" font-size="20.5" fill="#F9F7F4" font-weight="700">2a</text><rect x="460.1" y="1151.1" width="750.0" height="219.7" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="835.1" y="1203.7" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Pipeline notebooks,</text><text x="835.1" y="1256.4" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">visualization notebook, pytest</text><text x="835.1" y="1309.1" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">helper functions.</text><rect x="336.6" y="751.3" width="98.0" height="84.9" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="385.6" y="798.0" text-anchor="middle" font-size="20.5" fill="#F9F7F4" font-weight="700">2b</text><rect x="460.1" y="715.5" width="713.7" height="156.5" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="817.0" y="768.1" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Folder to separate unit tests</text><text x="817.0" y="820.8" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">from integration tests.</text><rect x="1746.5" y="919.8" width="98.0" height="84.9" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1795.5" y="966.5" text-anchor="middle" font-size="20.5" fill="#F9F7F4" font-weight="700">2b</text><rect x="1873.7" y="884.0" width="713.7" height="156.5" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="2230.6" y="936.6" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Folder that houses all</text><text x="2230.6" y="989.3" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">YAML files.</text><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s3" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s3" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s3" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s3" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="1137.6" y="908.6" width="915.1" height="219.7" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1595.1" y="961.2" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Additional granular folder structure to</text><text x="1595.1" y="1013.9" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">assist in organizing notebooks and</text><text x="1595.1" y="1066.6" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Python files.</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="384.8" y="1100.5" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="375.0" y="692.1" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="375.0" y="857.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="1014.3" y="980.0" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1058.7" y="1026.7" text-anchor="middle" font-size="40.0" fill="#F9F7F4" font-weight="700">3</text><path d="M1014.3,1018.5 L689.0,1018.5 L689.0,909.3 L363.6,909.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s3)"/><path d="M689.0,1018.5 L689.0,1197.0 L363.6,1197.0" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s3)"/><path d="M689.0,1018.5 L689.0,1376.3 L363.6,1376.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s3)"/><path d="M689.0,1018.5 L689.0,751.3 L376.1,751.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s3)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s4a" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s4a" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s4a" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s4a" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.8" y="1178.7" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="385.6" y="1250.8" text-anchor="middle" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="385.5" y="1506.6" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="375.1" y="939.5" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="374.1" y="758.4" text-anchor="middle" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="385.6" y="1422.4" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1213.8" y="896.3" width="915.1" height="156.5" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1671.3" y="948.9" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Notebooks and Python files that will</text><text x="1671.3" y="1001.6" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">be used in the Workflow.</text><rect x="1104.8" y="936.1" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1149.2" y="978.4" text-anchor="middle" font-size="18.8" fill="#F9F7F4" font-weight="700">4a</text><path d="M1104.8,974.6 L662.1,528.9" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4a)"/><path d="M1104.8,974.6 L609.0,754.8" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4a)"/><path d="M1104.8,974.6 L610.0,921.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4a)"/><path d="M1104.8,974.6 L619.7,1160.5" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4a)"/><path d="M1104.8,974.6 L620.5,1404.2" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4a)"/><path d="M1104.8,974.6 L634.9,1488.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4a)"/><path d="M1104.8,974.6 L620.5,1233.1" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4a)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s4b" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s4b" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s4b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s4b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="1879.1" y="1262.7" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1923.5" y="1304.7" text-anchor="middle" font-size="17.1" fill="#F9F7F4" font-weight="700">4b</text><rect x="2007.9" y="1222.9" width="915.1" height="156.5" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="2465.4" y="1275.5" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Required YAML file. This file hosts</text><text x="2465.4" y="1328.2" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">all top-level bundle mappings.</text><path d="M1879.1,1301.2 L1741.1,1301.2 L1741.1,1485.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnO-fpo2-s4b)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s5" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s5" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s5" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s5" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="768.0" y="1080.6" width="98.0" height="77.0" rx="2" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="772.6" y="1080.6" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="817.0" y="1127.3" text-anchor="middle" font-size="40.0" fill="#F9F7F4" font-weight="700">5</text><rect x="791.2" y="1179.6" width="915.1" height="156.5" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1248.8" y="1232.2" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Define the YAML files for the job</text><text x="1248.8" y="1284.9" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">and pipeline tasks.</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="717.7" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="964.3" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="1034.1" text-anchor="middle" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><path d="M968.5,699.5 L817.0,699.5 L817.0,1080.6" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-start="url(#cxnO-fpo2-s5)"/><path d="M968.5,1024.3 L861.4,1024.3 L861.4,1119.1" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-start="url(#cxnO-fpo2-s5)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s5a" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s5a" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s5a" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s5a" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="1094.8" y="796.4" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1139.2" y="838.7" text-anchor="middle" font-size="18.8" fill="#F9F7F4" font-weight="700">5a</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="717.7" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="1203.3" y="964.3" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="1034.1" text-anchor="start" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><path d="M1203.3,925.5 L1203.4,729.1" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s5a)"/><rect x="1223.1" y="775.4" width="980.1" height="126.2" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1713.1" y="819.0" text-anchor="middle" font-size="34.8" fill="#1B3139" font-weight="500">Use the pipeline ID created from the pipeline as a</text><text x="1713.1" y="859.1" text-anchor="middle" font-size="34.8" fill="#1B3139" font-weight="500">part of the task orchestrations in the job.yml file.</text><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s5b" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s5b" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s5b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s5b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="717.7" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="964.3" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="1034.1" text-anchor="middle" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><rect x="1869.3" y="1365.7" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1913.7" y="1407.7" text-anchor="middle" font-size="17.1" fill="#F9F7F4" font-weight="700">5b</text><rect x="1984.3" y="1351.2" width="956.3" height="106.1" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="2462.4" y="1387.2" text-anchor="middle" font-size="25.6" fill="#1B3139" font-weight="700">The job.yml and pipeline.yml files provide instructions for task</text><text x="2462.4" y="1416.6" text-anchor="middle" font-size="25.6" fill="#1B3139" font-weight="700">orchestration and must be included in the databricks.yml file.</text><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><path d="M1438.3,699.5 L1741.1,1485.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s5b)"/><path d="M1203.4,1053.9 L1741.1,1485.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s5b)"/><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s5c" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s5c" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s5c" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s5c" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="384.8" y="1100.5" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="385.5" y="1506.6" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="375.1" y="939.5" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="717.7" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="1180.6" y="796.4" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1225.0" y="838.7" text-anchor="middle" font-size="18.8" fill="#F9F7F4" font-weight="700">5c</text><rect x="1308.1" y="756.6" width="915.1" height="156.5" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1765.6" y="809.2" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">Use the notebooks and files defined</text><text x="1765.6" y="861.9" text-anchor="middle" font-size="45.8" fill="#1B3139" font-weight="500">previously to build the workflow</text><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><path d="M662.1,528.9 L968.5,699.5" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s5c)"/><path d="M610.0,921.3 L968.5,699.5" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s5c)"/><path d="M634.1,1177.0 L1203.4,729.1" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s5c)"/><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="964.3" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="1034.1" text-anchor="start" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><path d="M634.9,1488.4 L1203.4,729.1" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s5c)"/><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s6a" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s6a" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s6a" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s6a" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="717.7" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="964.3" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="1034.1" text-anchor="start" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><rect x="2137.8" y="836.9" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="2182.2" y="879.2" text-anchor="middle" font-size="18.8" fill="#F9F7F4" font-weight="700">6a</text><rect x="1500.0" y="797.1" width="610.5" height="141.4" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1805.2" y="845.1" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">Parameterize the variables</text><text x="1805.2" y="891.5" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">for the job.yml file.</text><rect x="2467.7" y="518.2" width="377.6" height="769.7" rx="28" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="2486.7" y="557.0" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">variables.yml</text><rect x="2522.7" y="591.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2645.8" y="639.3" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">my_email</text><rect x="2522.7" y="669.2" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2645.8" y="711.8" text-anchor="middle" font-size="33.6" fill="#0B2026" font-weight="500">target_catalog</text><rect x="2522.7" y="733.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="780.9" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">schema</text><rect x="2522.7" y="797.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2645.8" y="838.2" text-anchor="middle" font-size="31.8" fill="#0B2026" font-weight="500">raw_data_path</text><rect x="2522.7" y="863.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="911.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">username</text><rect x="2510.8" y="975.4" width="270.1" height="216.9" rx="28" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><rect x="2521.8" y="988.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1035.0" text-anchor="start" font-size="39.0" fill="#0B2026" font-weight="500">catalog_dev</text><rect x="2521.8" y="1053.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1095.8" text-anchor="start" font-size="33.9" fill="#0B2026" font-weight="500">catalog_stage</text><rect x="2521.8" y="1124.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1169.1" text-anchor="start" font-size="36.8" fill="#0B2026" font-weight="500">catalog_prod</text><rect x="2521.8" y="1205.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1252.8" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">cluster_id</text><path d="M2645.8,922.7 L2645.9,975.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6a)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><path d="M2522.7,621.1 L1438.3,699.5" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6a)"/><path d="M2522.7,698.8 L1438.3,699.5" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6a)"/><path d="M2522.7,826.7 L1438.3,699.5" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6a)"/><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s6b" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s6b" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s6b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s6b" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="717.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="964.3" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1203.4" y="1034.1" text-anchor="middle" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><rect x="1521.0" y="1031.0" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1565.4" y="1073.0" text-anchor="middle" font-size="17.1" fill="#F9F7F4" font-weight="700">6b</text><rect x="1628.2" y="998.8" width="610.5" height="141.4" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1933.4" y="1046.8" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">Parameterize the variables</text><text x="1933.4" y="1093.2" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">for the pipeline.yml file.</text><rect x="2467.7" y="518.2" width="377.6" height="769.7" rx="28" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="2486.7" y="557.0" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">variables.yml</text><rect x="2522.7" y="591.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="639.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">my_email</text><rect x="2522.7" y="669.2" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2645.8" y="711.8" text-anchor="middle" font-size="33.6" fill="#0B2026" font-weight="500">target_catalog</text><rect x="2522.7" y="733.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2645.8" y="780.9" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">schema</text><rect x="2522.7" y="797.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2645.8" y="838.2" text-anchor="middle" font-size="31.8" fill="#0B2026" font-weight="500">raw_data_path</text><rect x="2522.7" y="863.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="911.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">username</text><rect x="2510.8" y="975.4" width="270.1" height="216.9" rx="28" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><rect x="2521.8" y="988.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1035.0" text-anchor="start" font-size="39.0" fill="#0B2026" font-weight="500">catalog_dev</text><rect x="2521.8" y="1053.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1095.8" text-anchor="start" font-size="33.9" fill="#0B2026" font-weight="500">catalog_stage</text><rect x="2521.8" y="1124.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1169.1" text-anchor="start" font-size="36.8" fill="#0B2026" font-weight="500">catalog_prod</text><rect x="2521.8" y="1205.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1252.8" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">cluster_id</text><path d="M2645.8,922.7 L2645.9,975.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6b)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><path d="M2522.7,698.8 L1438.3,1024.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6b)"/><path d="M2522.7,762.7 L1438.3,1024.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6b)"/><path d="M2522.7,826.7 L1438.3,1024.3" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6b)"/><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s6c" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s6c" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s6c" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s6c" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="717.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="964.3" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="1034.1" text-anchor="start" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><rect x="2038.1" y="1260.9" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="2082.5" y="1302.9" text-anchor="middle" font-size="17.1" fill="#F9F7F4" font-weight="700">6c</text><rect x="1126.6" y="1173.6" width="891.8" height="252.5" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1572.5" y="1221.6" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">Parameterize the variables for the</text><text x="1572.5" y="1268.0" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">databricks.yml file. The variables.yml</text><text x="1572.5" y="1314.3" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">file must be included in the</text><text x="1572.5" y="1360.7" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">databricks.yml file.</text><rect x="2467.7" y="518.2" width="377.6" height="769.7" rx="28" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="2486.7" y="557.0" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">variables.yml</text><rect x="2522.7" y="591.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="639.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">my_email</text><rect x="2522.7" y="669.2" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="711.8" text-anchor="start" font-size="33.6" fill="#0B2026" font-weight="500">target_catalog</text><rect x="2522.7" y="733.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="780.9" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">schema</text><rect x="2522.7" y="797.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="838.2" text-anchor="start" font-size="31.8" fill="#0B2026" font-weight="500">raw_data_path</text><rect x="2522.7" y="863.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="911.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">username</text><rect x="2510.8" y="975.4" width="270.1" height="216.9" rx="28" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><rect x="2521.8" y="988.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1035.0" text-anchor="start" font-size="39.0" fill="#0B2026" font-weight="500">catalog_dev</text><rect x="2521.8" y="1053.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1095.8" text-anchor="start" font-size="33.9" fill="#0B2026" font-weight="500">catalog_stage</text><rect x="2521.8" y="1124.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1169.1" text-anchor="start" font-size="36.8" fill="#0B2026" font-weight="500">catalog_prod</text><rect x="2521.8" y="1205.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2645.0" y="1252.8" text-anchor="middle" font-size="40.0" fill="#0B2026" font-weight="500">cluster_id</text><path d="M2645.8,922.7 L2645.9,975.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6c)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><path d="M2510.8,1083.8 L2029.5,1515.0" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6c)"/><path d="M2521.8,1234.6 L2029.5,1515.0" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s6c)"/><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div>
<div id="fpo2-fr-s7" style="display:none"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA-fpo2-s7" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD-fpo2-s7" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO-fpo2-s7" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="713.9" y="496.8" width="2163.1" height="930.9" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="732.9" y="535.6" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1496.2" y="171.9" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="954.0" y="600.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="639.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="968.5" y="669.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="717.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="954.0" y="925.5" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="973.0" y="964.3" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="968.5" y="994.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="987.5" y="1034.1" text-anchor="start" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><rect x="1517.2" y="1374.6" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1561.6" y="1416.6" text-anchor="middle" font-size="17.1" fill="#F9F7F4" font-weight="700">7</text><rect x="853.7" y="1333.5" width="627.1" height="141.4" rx="2" fill="#EEEDE9" stroke="#1B3139" stroke-width="1.5"/><text x="1167.2" y="1381.5" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">Bring all relevant assets into</text><text x="1167.2" y="1427.9" text-anchor="middle" font-size="40.3" fill="#1B3139" font-weight="700">databricks.yml file.</text><rect x="2467.7" y="518.2" width="377.6" height="769.7" rx="28" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="2486.7" y="557.0" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">variables.yml</text><rect x="2522.7" y="591.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="639.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">my_email</text><rect x="2522.7" y="669.2" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="711.8" text-anchor="start" font-size="33.6" fill="#0B2026" font-weight="500">target_catalog</text><rect x="2522.7" y="733.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="780.9" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">schema</text><rect x="2522.7" y="797.1" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="838.2" text-anchor="start" font-size="31.8" fill="#0B2026" font-weight="500">raw_data_path</text><rect x="2522.7" y="863.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="911.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">username</text><rect x="2510.8" y="975.4" width="270.1" height="216.9" rx="28" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><rect x="2521.8" y="988.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1035.0" text-anchor="start" font-size="39.0" fill="#0B2026" font-weight="500">catalog_dev</text><rect x="2521.8" y="1053.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1095.8" text-anchor="start" font-size="33.9" fill="#0B2026" font-weight="500">catalog_stage</text><rect x="2521.8" y="1124.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1169.1" text-anchor="start" font-size="36.8" fill="#0B2026" font-weight="500">catalog_prod</text><rect x="2521.8" y="1205.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1252.8" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">cluster_id</text><path d="M2645.8,922.7 L2645.9,975.4" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA-fpo2-s7)"/><rect x="116.3" y="391.9" width="2760.8" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1496.7" y="451.7" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="108.1" y="989.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="1028.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="108.1" y="570.4" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="127.1" y="609.2" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="135.4" y="1061.7" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.4" y="1100.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="135.2" y="1308.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="154.2" y="1347.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="125.7" y="653.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="692.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="125.7" y="819.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="144.7" y="857.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="149.9" y="1130.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="168.9" y="1178.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="150.7" y="1203.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1250.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="136.2" y="1458.8" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="155.2" y="1506.6" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="140.2" y="891.7" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="159.2" y="939.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="139.2" y="725.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="158.2" y="758.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="150.7" y="1374.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="169.7" y="1422.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="107.1" y="499.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="384.6" y="534.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><rect x="1452.7" y="1485.4" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1741.1" y="1521.0" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><text x="1499.9" y="329.2" text-anchor="middle" font-size="89.0" fill="#FF3621" font-weight="700">Folder Architecture</text></svg></div></div><div style="text-align:center;max-width:1180px;margin:16px auto 0"><div id="fpo2-btn-s1" onclick="fpo2_go('s1')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s1" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">1</span><span style="vertical-align:middle">Create the project folder</span></div><div id="fpo2-btn-s2" onclick="fpo2_go('s2')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s2" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">2</span><span style="vertical-align:middle">Add high-level directories</span></div><div id="fpo2-btn-s2b" onclick="fpo2_go('s2b')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s2b" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">2a / 2b</span><span style="vertical-align:middle">Purpose of each directory</span></div><div id="fpo2-btn-s3" onclick="fpo2_go('s3')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s3" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">3</span><span style="vertical-align:middle">Add granular sub-folders</span></div><div id="fpo2-btn-s4a" onclick="fpo2_go('s4a')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s4a" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">4a</span><span style="vertical-align:middle">Add notebooks & Python files</span></div><div id="fpo2-btn-s4b" onclick="fpo2_go('s4b')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s4b" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">4b</span><span style="vertical-align:middle">Add databricks.yml</span></div><div id="fpo2-btn-s5" onclick="fpo2_go('s5')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s5" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">5</span><span style="vertical-align:middle">Define job & pipeline YAML</span></div><div id="fpo2-btn-s5a" onclick="fpo2_go('s5a')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s5a" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">5a</span><span style="vertical-align:middle">Pipeline ID in job.yml</span></div><div id="fpo2-btn-s5b" onclick="fpo2_go('s5b')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s5b" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">5b</span><span style="vertical-align:middle">Include YAML in databricks.yml</span></div><div id="fpo2-btn-s5c" onclick="fpo2_go('s5c')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s5c" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">5c</span><span style="vertical-align:middle">Files used inside job.yml</span></div><div id="fpo2-btn-s6a" onclick="fpo2_go('s6a')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s6a" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">6a</span><span style="vertical-align:middle">Variables for job.yml</span></div><div id="fpo2-btn-s6b" onclick="fpo2_go('s6b')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s6b" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">6b</span><span style="vertical-align:middle">Variables for pipeline.yml</span></div><div id="fpo2-btn-s6c" onclick="fpo2_go('s6c')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s6c" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">6c</span><span style="vertical-align:middle">Variables for databricks.yml</span></div><div id="fpo2-btn-s7" onclick="fpo2_go('s7')" style="display:inline-block;vertical-align:middle;margin:4px;padding:7px 14px 7px 9px;border:1.5px solid #D5D2CC;border-radius:22px;background:#fff;cursor:pointer;font-family:DM Sans,system-ui,sans-serif;font-size:13.5px;font-weight:600;color:#0B2026;user-select:none;line-height:22px;white-space:nowrap"><span id="fpo2-bd-s7" style="display:inline-block;vertical-align:middle;min-width:12px;height:22px;padding:0 6px;margin-right:8px;border-radius:11px;background:#FF5F46;color:#fff;font-size:12px;font-weight:700;text-align:center;line-height:22px">7</span><span style="vertical-align:middle">Assemble databricks.yml</span></div></div><div style="max-width:1180px;margin:16px auto 0;background:#fff;border:1px solid #EEEDE9;border-left:5px solid #1B5162;border-radius:10px;padding:16px 22px"><div id="fpo2-nt-s1" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 1 — Create the project folder</div><div style="font-size:15px;line-height:1.6;color:#0B2026">The first step in setting up a Declarative Automation Bundle is to create a structured project directory.</div></div>
<div id="fpo2-nt-s2" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 2 — Add high-level directories</div><div style="font-size:15px;line-height:1.6;color:#0B2026">At the root level, the project follows this layout: <code>Full Project/</code> with <code>resources/</code>, <code>src/</code>, <code>tests/</code>, and <code>databricks.yml</code>. High-level directories isolate the project components.</div></div>
<div id="fpo2-nt-s2b" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 2a / 2b — Purpose of each directory</div><div style="font-size:15px;line-height:1.6;color:#0B2026">Each directory serves a specific purpose: <code>resources/</code> stores YAML configuration files for workflows and pipelines; <code>src/</code> contains all source code, such as Python and SQL notebooks; <code>tests/</code> includes unit tests and integration tests for validation; and <code>databricks.yml</code> is the required configuration file that defines the Declarative Automation Bundle. This structure keeps code, configurations, and tests organized and easy to manage.</div></div>
<div id="fpo2-nt-s3" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 3 — Add granular sub-folders</div><div style="font-size:15px;line-height:1.6;color:#0B2026">Testing is a crucial part of CI/CD, ensuring that code works before deployment. Unit tests validate individual components like functions or transformations. Integration tests ensure that different Databricks assets (jobs, workflows, pipelines) work together as expected. All tests are run automatically in the CI pipeline to catch errors early. The <code>src/</code> directory holds the actual implementation logic for the project.</div></div>
<div id="fpo2-nt-s4a" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 4a — Add notebooks & Python files</div><div style="font-size:15px;line-height:1.6;color:#0B2026">Once we have our folder structure for <code>src/</code> and <code>tests/</code>, we build out the notebooks and Python files that will be used with this project.</div></div>
<div id="fpo2-nt-s4b" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 4b — Add databricks.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026">Next, we will add the <code>databricks.yml</code> file that hosts all top-level bundle mappings for our project. The <code>databricks.yml</code> file is the core configuration of the Declarative Automation Bundle. It includes the bundle name, resources (workflows, pipelines, models), targets (development, staging, production), and variables (paths, authentication details).</div></div>
<div id="fpo2-nt-s5" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 5 — Define job & pipeline YAML</div><div style="font-size:15px;line-height:1.6;color:#0B2026">Next, let’s take a look at the resources folder. The <code>resources/</code> directory is where all YAML configuration files for Declarative Automation Bundles are stored.</div></div>
<div id="fpo2-nt-s5a" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 5a — Pipeline ID in job.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026"><code>dabs_workflow.job.yml</code> defines job tasks, execution order, and dependencies. <code>health_etl_pipeline.pipeline.yml</code> defines DLT pipelines, specifying transformations and output tables.</div></div>
<div id="fpo2-nt-s5b" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 5b — Include YAML in databricks.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026">We include the job and pipeline YAML files in the <code>databricks.yml</code> file.</div></div>
<div id="fpo2-nt-s5c" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 5c — Files used inside job.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026">We see here all the files and folders that are to be used inside the <code>job.yml</code> file — this brings together the complete workflow that will be deployed and run within Databricks.</div></div>
<div id="fpo2-nt-s6a" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 6a — Variables for job.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026">To help modularize the setup for the demo, we have separated the variables from the <code>databricks.yml</code> file. When you edit the configurations for the bundle, you will do so by manipulating the <code>databricks.yml</code> file or the <code>variables.yml</code> file — each have their own purpose.</div></div>
<div id="fpo2-nt-s6b" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 6b — Variables for pipeline.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026">Similarly, we configure any variable that is to be used in the <code>pipeline.yml</code> file as well.</div></div>
<div id="fpo2-nt-s6c" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 6c — Variables for databricks.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026">The <code>databricks.yml</code> file is the core configuration YAML file for high-level orchestration. The <code>variables.yml</code> file is used to fine-tune the bundle configuration, like defining the default user and compute cluster.</div></div>
<div id="fpo2-nt-s7" style="display:none"><div style="font-weight:700;font-size:15px;color:#1B5162;margin-bottom:6px">Step 7 — Assemble databricks.yml</div><div style="font-size:15px;line-height:1.6;color:#0B2026">Note that we can certainly overwrite the default variable values within the <code>databricks.yml</code> file. Once all bundle assets have been assembled, we are ready to validate, deploy, and run the bundle using the Databricks CLI.</div></div></div></div><script>(function(){var K=['s1', 's2', 's2b', 's3', 's4a', 's4b', 's5', 's5a', 's5b', 's5c', 's6a', 's6b', 's6c', 's7'];window.fpo2_go=function(sel){for(var i=0;i<K.length;i++){var k=K[i];var fr=document.getElementById("fpo2-fr-"+k);if(fr)fr.style.display=(k===sel?"block":"none");var nt=document.getElementById("fpo2-nt-"+k);if(nt)nt.style.display=(k===sel?"block":"none");var bt=document.getElementById("fpo2-btn-"+k);var bd=document.getElementById("fpo2-bd-"+k);if(bt){if(k===sel){bt.style.background="#1B5162";bt.style.borderColor="#1B5162";bt.style.color="#fff";if(bd){bd.style.background="#fff";bd.style.color="#1B5162";}}else{bt.style.background="#fff";bt.style.borderColor="#D5D2CC";bt.style.color="#0B2026";if(bd){bd.style.background="#FF5F46";bd.style.color="#fff";}}}}};fpo2_go("s1");})();</script>

### B2. Full Project Outline — Summary

To summarize, this demonstration covers the folder structure of a Declarative Automation Bundle, how source code, tests, and configurations are organized, and how to validate, deploy, and run a bundle using the CLI. By following this structure, teams can efficiently manage, test, and deploy Databricks projects in a reliable CI/CD pipeline. The numbered steps below map to the build-up shown in B1.

<div style="font-family:DM Sans,system-ui,sans-serif;max-width:1200px;margin:8px auto"><div style="text-align:center;max-width:1000px;margin:18px auto 6px"><div style="font-size:32px;font-weight:700;color:#0B2026">Full Project Outline</div><div style="font-size:22px;color:#1B5162;font-weight:500;margin-top:2px">Summary</div></div><div style="border:1px solid #E2E0DB;border-radius:12px;background:#fff;padding:10px 14px;max-width:1180px;margin:0 auto"><svg viewBox="0 388 3000 1300" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="" font-family="DM Sans,system-ui,-apple-system,Segoe UI,Roboto,sans-serif" style="width:100%;height:auto;display:block;"><defs><marker id="cxnA" viewBox="0 0 10 10" refX="8.5" refY="5" markerWidth="6.5" markerHeight="6.5" orient="auto-start-reverse"><path d="M0,0 L10,5 L0,10 z" fill="context-stroke"/></marker><marker id="cxnD" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5.5" markerHeight="5.5" orient="auto"><path d="M5,0 L10,5 L5,10 L0,5 z" fill="context-stroke"/></marker><marker id="cxnO" viewBox="0 0 10 10" refX="5" refY="5" markerWidth="5" markerHeight="5" orient="auto"><circle cx="5" cy="5" r="4.5" fill="context-stroke"/></marker></defs><rect x="1787.4" y="509.3" width="1089.7" height="811.8" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1806.4" y="548.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">resources/</text><text x="1499.9" y="301.1" text-anchor="middle" font-size="60.3" fill="#1B5162" font-weight="500">Summary</text><text x="1496.2" y="184.4" text-anchor="middle" font-size="89.0" fill="#0B2026" font-weight="700">Full Project Outline</text><rect x="2010.6" y="1416.1" width="576.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="2299.0" y="1451.7" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">databricks.yml</text><rect x="1827.8" y="613.7" width="530.9" height="347.1" rx="28" fill="none" stroke="#FF5F46" stroke-width="1.5"/><rect x="1841.7" y="633.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="1860.7" y="672.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">job</text><rect x="1856.2" y="702.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1875.2" y="750.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">dabs_workflow.job.yml</text><rect x="1844.7" y="801.7" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="1863.7" y="840.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">pipeline</text><rect x="1859.2" y="871.0" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1878.2" y="910.4" text-anchor="start" font-size="29.8" fill="#0B2026" font-weight="500">health_etl_pipeline.pipeline.yml</text><rect x="2467.7" y="530.7" width="377.6" height="769.7" rx="28" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="2486.7" y="569.5" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">variables.yml</text><rect x="2522.7" y="604.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="651.8" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">my_email</text><rect x="2522.7" y="681.7" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="724.3" text-anchor="start" font-size="33.6" fill="#0B2026" font-weight="500">target_catalog</text><rect x="2522.7" y="745.6" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="793.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">schema</text><rect x="2522.7" y="809.6" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="850.7" text-anchor="start" font-size="31.8" fill="#0B2026" font-weight="500">raw_data_path</text><rect x="2522.7" y="876.0" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2541.7" y="923.8" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">username</text><rect x="2510.8" y="987.9" width="270.1" height="216.9" rx="28" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><rect x="2521.8" y="1000.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1047.5" text-anchor="start" font-size="39.0" fill="#0B2026" font-weight="500">catalog_dev</text><rect x="2521.8" y="1065.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1108.3" text-anchor="start" font-size="33.9" fill="#0B2026" font-weight="500">catalog_stage</text><rect x="2521.8" y="1136.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1181.6" text-anchor="start" font-size="36.8" fill="#0B2026" font-weight="500">catalog_prod</text><rect x="2521.8" y="1217.5" width="246.3" height="59.2" rx="8.3" fill="#D9D9D9" stroke="#FF5F46" stroke-width="1.5"/><text x="2540.8" y="1265.3" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">cluster_id</text><path d="M2645.8,935.2 L2645.9,987.9" fill="none" stroke="#FF5F46" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round" marker-end="url(#cxnA)"/><rect x="1184.7" y="404.4" width="1692.4" height="95.6" rx="13.4" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="2030.9" y="464.2" text-anchor="middle" font-size="58.7" fill="#F9F7F4" font-weight="700">Full Project/</text><rect x="1185.7" y="999.3" width="555.0" height="543.3" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1204.7" y="1038.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">src/</text><rect x="1185.7" y="580.3" width="555.0" height="409.6" rx="28" fill="#1B5162" stroke="#1B5162" stroke-width="1.5"/><text x="1204.7" y="619.1" text-anchor="start" font-size="29.0" fill="#F9F7F4" font-weight="700">tests/</text><rect x="1213.0" y="1071.6" width="498.7" height="230.5" rx="28" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="1232.0" y="1110.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">dlt_pipelines</text><rect x="1212.8" y="1318.6" width="498.7" height="133.7" rx="18.7" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="1231.8" y="1357.4" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">helpers</text><rect x="1203.3" y="663.3" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="1222.3" y="702.1" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">unit_tests</text><rect x="1203.3" y="829.1" width="498.7" height="143.6" rx="20.1" fill="#FFAB00" stroke="#FFAB00" stroke-width="1.5"/><text x="1222.3" y="867.9" text-anchor="start" font-size="29.0" fill="#0B2026" font-weight="500">integration_test</text><rect x="1227.5" y="1140.9" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1246.5" y="1188.7" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">gold_tables_dlt</text><rect x="1228.3" y="1213.5" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1247.3" y="1260.8" text-anchor="start" font-size="39.4" fill="#0B2026" font-weight="500">ingest-bronze-silver_dlt</text><rect x="1213.8" y="1468.7" width="498.7" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1232.8" y="1516.5" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">Final Visualization</text><rect x="1217.8" y="901.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1236.8" y="949.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">integration_tests_dlt</text><rect x="1216.8" y="735.2" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1235.8" y="768.4" text-anchor="start" font-size="22.2" fill="#0B2026" font-weight="500">test_spark_helper_functions.py</text><rect x="1228.3" y="1384.6" width="469.8" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1247.3" y="1432.4" text-anchor="start" font-size="40.0" fill="#0B2026" font-weight="500">project_functions.py</text><rect x="1184.7" y="509.3" width="555.0" height="59.2" rx="8.3" fill="#EEEDE9" stroke="#FF5F46" stroke-width="1.5"/><text x="1462.2" y="544.9" text-anchor="middle" font-size="29.0" fill="#0B2026" font-weight="500">run_unit_tests</text><text x="169.0" y="443.8" text-anchor="start" font-size="66.0" fill="#1B3139" font-weight="500">CI/CD with DABs Summary</text><rect x="1170.5" y="370.6" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1214.9" y="416.2" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">1</text><rect x="1686.6" y="983.7" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1731.0" y="1029.3" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">2</text><rect x="1679.2" y="573.6" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1723.6" y="619.2" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">3</text><rect x="1686.6" y="469.0" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="1731.0" y="514.6" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">4</text><rect x="2280.5" y="586.3" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="2324.9" y="631.9" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">5</text><rect x="2802.5" y="546.8" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="2846.9" y="592.4" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">6</text><rect x="1986.4" y="1375.7" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="2030.8" y="1421.3" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">7</text><text x="242.0" y="599.5" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">Define root directory for the project</text><text x="242.0" y="699.9" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">Incorporate tested notebooks - these</text><text x="242.0" y="742.1" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">notebooks should be tested in isolation</text><text x="242.0" y="784.3" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">prior to migrating to DABs</text><text x="242.0" y="901.2" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">Set up a folder for isolating different</text><text x="242.0" y="943.4" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">tests, e.g. unit tests and integration tests</text><text x="242.0" y="1052.1" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">Define any other notebooks</text><text x="242.0" y="1152.4" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">Define YAML files for jobs and pipeline</text><text x="242.0" y="1194.6" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">tasks</text><text x="242.0" y="1303.3" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">Parameterize all YAML files with</text><text x="242.0" y="1345.5" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">variables.yml if possible</text><text x="242.0" y="1454.1" text-anchor="start" font-size="36.7" fill="#1B3139" font-weight="500">Define databricks.yml file</text><rect x="123.1" y="556.4" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="167.5" y="602.0" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">1</text><rect x="123.1" y="696.6" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="167.5" y="742.2" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">2</text><rect x="123.1" y="883.3" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="167.5" y="928.9" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">3</text><rect x="123.1" y="1019.1" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="167.5" y="1064.7" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">4</text><rect x="123.1" y="1154.9" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="167.5" y="1200.5" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">5</text><rect x="123.1" y="1290.7" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="167.5" y="1336.3" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">6</text><rect x="123.1" y="1411.0" width="88.8" height="77.0" rx="2" fill="#FF5F46" stroke="#FF5F46" stroke-width="1.5"/><text x="167.5" y="1456.6" text-anchor="middle" font-size="34.8" fill="#F9F7F4" font-weight="700">7</text></svg></div></div>

## C. Inspect Pre-Configured YAML Files

Our goal is to deploy our project to the `dev`, `stage`, and `prod` environments for our CI/CD pipeline. 

In this example, the project is a simple workflow that contains unit tests, a Spark Declarative Pipeline, and a notebook visualization.

![Workflow](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/13 Demo - Continuous Integration and Continuous Deployment with DABs/images/06_Final_Workflow_Desc.png)



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Prerequisites
  </strong>
  <div style="color:#333;">

This advanced-level course assumes prerequisite knowledge of essential DevOps concepts such as code modularization, custom Python functions, unit testing with pytest, and integration tests with Spark Declarative Pipelines. 

For a refresher on those topics, see the Databricks course **DevOps Essentials for Data Engineering**. We touch on each here, but the focus of this course is deployment with Declarative Automation Bundles.

  </div>
</div>



Let's explore our project folder called **Full Project**. This folder contains all of our Databricks resources to deploy.

You will find the following in the root folder:

  - **src/**
  - **resources/**
  - **databricks.yml**
  - **tests/**

1. In a new tab, open the **databricks.yml** file.

    - It begins by defining the bundle name under the `bundle` mapping.
        - **health_etl_bundle**

    - It defines the resources to include under the `include` mapping. 
        - All the YAML configuration resource files are in the **resources/** folder.

    - Under the `targets` top-level mapping, you will see three defined targets and a variety of configuration specifics for each: `development`, `stage`, and `production`.
        - All three targets have a `root_path`.
        - All three targets have specific configurations.
        - The `stage` and `production` targets have additional variables we need to configure, such as `target_catalog` and `raw_data_path`, to specify the correct data.


2. In the new tab, open **resources/**.

    - Click on the YAML file named **variables.yml**. 
        - It contains pre-defined variables for the resources to be deployed when deploying the bundle. 
        - These include things like the job name, notebook paths, and parameters to pass to the notebooks.

    - You will find two folders for a job and SDP resource: 
        - **job/**
           - The **dabs_workflow.job.yml** file (located in the **job/** folder) describes the tasks that will be created. 
           - Notice that there are 3 tasks: **Unit_Tests**, **Visualization**, and **Health_ETL**. 
           - While **Health_ETL** is listed after **Visualization**, it depends on **Unit_Tests**. 
           - The order of the tasks doesn't matter, since the `depends_on` key configures the dependencies.

        - **pipeline/**
            - The **health_etl_pipeline.pipeline.yml** file (located in the **pipeline/** folder) describes the Spark Declarative Pipeline configuration.


3. In the new tab, navigate back to **src/** in the root folder. 

    This folder contains other folders and notebooks that are called from the YAML files you inspected in the previous steps. These notebooks are chained together as part of the workflow we will deploy below.

    - **dlt_pipelines/**: contains two Spark Declarative Pipeline notebooks:
      - **gold_tables_dlt**
      - **ingests-bronze-silver_dlt** 
      - You can inspect these notebooks to understand their role in the **Health_ETL** workflow.

    - **Final Visualization**: this notebook is the final task in our workflow. 
      - It creates a stacked bar chart of cholesterol distribution by age group.

    - **helpers/**: contains a `.py` file with custom Python methods for the transformation of the data in the pipeline.

## D. Explore and Update YAML Configuration Files
We will update our YAML files to better understand how to point to the assets and variables needed to configure the bundle before validation using the Databricks CLI.

### D1. Explore the databricks.yml Configuration

Recall that to use a variable called `my_variable` in a bundle, refer to it using `${var.my_variable}`.

#### Instructions

1. Navigate to the folder named **Full Project**.

2. Click on the file **databricks.yml** and explore the bundle configuration.

3. Locate the mapping **targets**. 
   - Each target is a unique collection of artifacts, Databricks workspace settings, and Databricks job or pipeline details.
   - The targets mapping consists of one or more target mappings, which must each have a unique programmatic (or logical) name.

4. Locate the **development** target and examine the configuration. Notice the following:
   - The value for `default` is set to `True`.
   - The value for `existing_cluster_id` uses the variable `cluster_id`.
   - The **tasks** are set to use our lab compute cluster.

5. Locate the **stage** target and examine the configuration. Notice the following:
   - The `target_catalog` variable uses the variable `catalog_stage`.
   - The `raw_data_path` variable uses the volume `health` in `catalog_stage`.
   - The **tasks** are set to use our lab compute cluster.

6. Locate the **production** target and examine the configuration. Notice the following:
   - The `target_catalog` variable uses the variable `catalog_prod`.
   - The `raw_data_path` variable uses the volume `health` in `catalog_prod`.
   - No compute cluster is specified for the job. The default compute will use Serverless in production.


In [0]:
print(f'Your user name: {my_catalog}')

### D2. Update `variables.yml`

Next, we will update the file **variables.yml**.

#### Instructions

1. Navigate to the **resources/** folder.

2. Click on the file **variables.yml**.

3. Fill in the following details for the variables:

   - **TO DO**: `username`: Add your username here. Your username can be found in the cell above.
      - Use `${workspace.current_user.short_name}`

   - **TO DO**: `my_email`: Enter your email address here. This is used to send notifications.
      - Use your email address.

   - **TO DO**: `cluster_id`:
      - Use the `lookup` function with your username to obtain the cluster ID value.
      - Paste the value from the cell above for the lookup cluster ID variable.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">


- The file **variables_solution.yml** contains an example solution if you need help.

- In the Databricks Academy lab environment, all catalogs, clusters, and usernames match and have no spaces by default.

  </div>
</div>


## E. Visualizing the Declarative Automation Bundle's Assets

Here we'll look at how to manually update our YAML files to help get acquainted with the setup. 

Since we are bringing in a pre-configured bundle, it's worth looking at the structure of files we'll be interacting with. Below is a diagram representing how the variables for the development catalog will be used.

![Full Pipeline](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/13 Demo - Continuous Integration and Continuous Deployment with DABs/images/06_img1.png)

## F. Notebook Execution

Now that we are familiar with the various folders and files that make up our bundle, let's make sure the CLI is installed by authenticating.

### F1. Development Bundle

Here is what the configuration of our target mapping for development looks like in the databricks.yml file. 
```YAML
targets:

  development:
    mode: development
    default: true
    # In Development, we will use classic compute for our tasks 
    resources:
      jobs:
        health_etl_workflow:    
          name: health_etl_workflow_${bundle.target} 
          tasks:
            - task_key: Unit_Tests
              existing_cluster_id: ${var.cluster_id}
            - task_key: Visualization
              existing_cluster_id: ${var.cluster_id}
    workspace:
      root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}
...
```

**NOTE:** Recall that with **development** and **stage** target environments we are using a mix of serverless and classic compute at the task level.

1. To validate the bundle, run the following cell. 

    This uses all the default values from **variables.yml** (see diagram above).

In [0]:
%sh 
cd "Full Project" 
pwd;
databricks bundle validate -t development

<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
     Troubleshooting
  </strong>
  <div style="color:#333;">
If you see the following error after validating your bundle, the format of your notebook could be incorrect.

`Error: notebook xxx.ipynb not found`. 

Check the format of your notebook and adjust accordingly. 

  </div>
</div>



2. After the development target validates, deploy the bundle to the development environment.

In [0]:
%sh
cd "Full Project" 
databricks bundle deploy -t development

3. Navigate to **Jobs & Pipelines** in a new tab and view your deployed job.
    - Leave this tab open.

#### Checkpoint
![Dev CICD Pipeline](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/cicd-pipeline/dev-deployed-job.png)

4. To run the bundle using the Databricks CLI, run the following cell. Note that the job will show as **[dev <username>] health_etl_workflow_<target>** within **Jobs and Pipelines**.

    This makes sense when you refer back to the structure of the **dabs_workflow.job.yml** file located in **resources/**:

    ```yaml
    resources:
      jobs:
        health_etl_workflow:                          # <--- Job key (used by `bundle run`)
          name: health_etl_workflow_${bundle.target}  # <--- Job name (shown in the UI)
          description: Final Workflow SDK
    ```

In [0]:
%sh
cd "Full Project" 
databricks bundle run health_etl_workflow

5. Navigate back to your Job and view the successful run on development.

#### Checkpoint - Dev Run
![Dev CICD Pipeline](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/cicd-pipeline/dev-job-run.png)



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Summary - Development
  </strong>
  <div style="color:#333;">

While the job is running, examine the tasks when using the `development` target. 

  Note the following:
- Unit tests passed.
- The Spark Declarative Pipeline ETL and integration tests passed on a small sample of 7,500 rows of dev data.
- The visualization was created using the small sample of 7,500 rows of dev data.
  </div>
</div>

### F2. Staging Bundle

Here is what the configuration of our `targets` mapping for `stage` looks like in **databricks.yml**:

```yaml
  ...

  stage:
    mode: development
      # In stage, we use classic compute for our tasks
    resources:
      jobs:
        health_etl_workflow:
          name: health_etl_workflow_${bundle.target}
          tasks:
            - task_key: Unit_Tests
              existing_cluster_id: ${var.cluster_id}
            - task_key: Visualization
              existing_cluster_id: ${var.cluster_id}
    workspace:
      root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}
    variables:
      target_catalog: ${var.catalog_stage}
      raw_data_path: /Volumes/${var.catalog_stage}/default/health
```

</br>

Imagine you've reviewed your code, analyzed coverage, and so on, and you're ready to deploy and test in a staging environment. 

DABs simplifies this by adjusting a few parameter values. Run the following cells to validate, deploy, and run with `stage` as the target.

</br>

In this example, since `target_catalog` and `raw_data_path` have default values, we override them when deploying to other targets like `stage` within the `targets` mapping. This makes the job read data from the staging catalog.



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Bonus Variable Overrides
  </strong>
  <div style="color:#333;">

You can also override variable values directly through the Databricks CLI. For example: `databricks bundle validate --var="target_catalog=<username>_2_stage" -t stage`. Keep in mind this will just reproduce the same job you just ran.

  </div>
</div>


In [0]:
%sh
cd "Full Project" 
databricks bundle validate -t stage

In [0]:
%sh
cd "Full Project" 
databricks bundle deploy -t stage

In [0]:
%sh
cd "Full Project" 
databricks bundle run health_etl_workflow -t stage

#### Checkpoint - Stage Run
![Stage CICD Pipeline](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/cicd-pipeline/stage-job-run.png)


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Summary - Stage
  </strong>
  <div style="color:#333;">

While the job is running, open the staged job (**[dev labuser_UNIQUE_ID] health_etl_workflow_stage**)and examine the tasks when using the `stage` target. 

  Note the following:
- Unit tests passed.
- The Spark Declarative Pipeline ETL and integration tests passed on a sample of 35,000 rows of stage data.
- The visualization was created using the sample of 35,000 rows of stage data.
  </div>
</div>


### F3. Production
Here is what the configuration of our target mapping for production looks like in the **databricks.yml** file.
```YAML
  production:
    mode: production
    workspace:
      # host: can change host if isolating by workspace
      root_path: /Workspace/Users/${workspace.current_user.userName}/.bundle/${bundle.name}/${bundle.target}
    variables:
      target_catalog: ${var.catalog_prod}
      raw_data_path: /Volumes/${var.catalog_prod}/default/health
```

Here, we'll repeat the same bash commands using `%sh`. 

However, note that all production compute will run on **serverless instead of classic compute**, as we're not overriding the default compute.

You can verify this by deploying the job and inspecting the tasks.

In [0]:
%sh
cd "Full Project" 
databricks bundle validate -t production

In [0]:
%sh
cd "Full Project" 
databricks bundle deploy -t production

In [0]:
%sh
cd "Full Project" 
databricks bundle run health_etl_workflow -t production

#### Checkpoint - Production Run
![Prod CICD Pipeline](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/Includes/images/cicd-pipeline/prod-job-run.png)

## G. Destroy All Bundles

Now that we've validated, deployed, and run the bundle against all three targets, clean up by destroying each one. The cell below destroys `development`, `stage`, and `production` in sequence.

In [0]:
%sh
cd "Full Project";
databricks bundle destroy -t development --auto-approve;
databricks bundle destroy -t stage --auto-approve;
databricks bundle destroy -t production --auto-approve;

## Conclusion

In this demonstration you walked an end-to-end CI/CD-style workflow using a single bundle promoted across three targets:

1. Inspected a real-world bundle that splits resources into per-asset YAML files (`job/`, `pipeline/`) and a dedicated `variables.yml`.
2. Set the `username`, `my_email`, and `cluster_id` (lookup) variables so the bundle resolves correctly for your lab.
3. Validated, deployed, and ran the workflow against the `development` target (7,500 rows on classic compute).
4. Promoted the same bundle to the `stage` target (35,000 rows on classic compute) by overriding `target_catalog` and `raw_data_path`.
5. Promoted the same bundle to the `production` target (full dataset on Serverless compute).
6. Cleaned up by destroying all three targets with `databricks bundle destroy --auto-approve` per target.

## Next Steps

![ci_cd](https://files.training.databricks.com/binder/prod_main/automated-deployment-with-declarative-automation-bundles-en_us-2.4.0/images/20260828T161450Z/Automated Deployment with Declarative Automation Bundles/13 Demo - Continuous Integration and Continuous Deployment with DABs/images/ci_cd_overview.png)

Think about how you can use DABs to accelerate development by programmatically managing your workflows. With DABs you can create, manage, and deploy your different assets and artifacts in a consistent and repeatable manner for CI/CD workflows.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>